# Expanded Q4 2025 AI/Tech Earnings Eval

This notebook runs or loads cached outputs for the expanded AI/Tech Q4 2025 earnings universe. Agent outputs are saved under `data/agent_outputs/q4_2025_ai_tech/` by agent type, so future runs can reuse them without calling the models again.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env", override=True)

from openclam.evaluation import q4_earnings_cache as q4
from openclam.agents.news_macro import news_macro_agent

print(f"Repo root: {REPO_ROOT}")
print(f"Cache root: {REPO_ROOT / q4.DEFAULT_CACHE_ROOT}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")


## Configure Universe and Cache

Set `RUN_AGENTS=True` when you want to generate missing cached outputs. Keep `FORCE_RERUN=False` to reuse files that already exist.


In [ ]:
STARTER_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA",
    "AMD", "AVGO", "MU", "TSM", "ASML", "AMAT", "LRCX",
    "ORCL", "DELL", "ANET", "VRT", "CRM", "PLTR",
]

CACHE_ROOT = REPO_ROOT / q4.DEFAULT_CACHE_ROOT
RUN_AGENTS = True
FORCE_RERUN = False

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

earnings_df = q4.q4_2025_ai_tech_earnings_df(STARTER_TICKERS)
earnings_df


## Build Price Eval Table


In [ ]:
summary_df, paths_df = q4.build_price_tables(
    earnings_df=earnings_df,
    cache_root=CACHE_ROOT,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

news_macro_agent.format_return_columns(summary_df)


## Run or Load Cached Agent Outputs


In [ ]:
if RUN_AGENTS:
    packets_by_ticker, errors_by_ticker = q4.run_q4_2025_universe_agents(
        earnings_df=earnings_df,
        cache_root=CACHE_ROOT,
        force=FORCE_RERUN,
        lookback_days=14,
        max_news=10,
        llm_provider=LLM_PROVIDER,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
        news_model=NEWS_MODEL,
    )
else:
    packets_by_ticker = q4.load_cached_packets_by_ticker(CACHE_ROOT)
    errors_by_ticker = {}

print("cached/available tickers:", sorted(packets_by_ticker))
errors_by_ticker


## Run CIO Eval From Cached Packets


In [ ]:
cio_eval, cio_results, cio_summary = q4.run_cached_cio_eval(
    summary_df,
    packets_by_ticker,
    cache_root=CACHE_ROOT,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    use_llm_debate=True,
    use_llm_decision=True,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)

news_macro_agent.format_return_columns(cio_eval)


In [ ]:
cio_summary


## Inspect Saved Outputs


In [ ]:
ticker_to_inspect = "NVDA"
cached = q4.load_cached_ticker(ticker_to_inspect, CACHE_ROOT)
print("saved files:")
for key, value in q4.cached_ticker_paths(ticker_to_inspect, CACHE_ROOT).items():
    print(key, value)

packets = cached.get("packets") or []
packet_df = pd.DataFrame(packets)
display_cols = ["agent_name", "short_term_stance", "long_term_stance", "confidence", "stance_rationale"]
packet_df[[col for col in display_cols if col in packet_df.columns]]


In [ ]:
# Debate trace for one ticker
workflow = cio_results.get(ticker_to_inspect) or q4.load_json(CACHE_ROOT / "cio" / f"{ticker_to_inspect}.json")
pd.DataFrame(workflow["debate"].get("debate_responses", []))
